In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: Isang Qiskit Function ng Qedma
> **Note:** Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status pa ito at maaaring magbago.

## Pangkalahatang-ideya
Kahit na malaki na ang pagbabago ng mga quantum processing unit sa mga nakaraang taon, ang mga error dahil sa ingay at mga kakulangan sa kasalukuyang hardware ay nananatiling pangunahing hamon para sa mga developer ng quantum algorithm. Habang papalapit ang larangan sa mga utility-scale na quantum computation na hindi na mapapatunayan nang klasikal, nagiging mas mahalaga ang mga solusyon para sa pagkansela ng ingay na may garantisadong katumpakan. Para malampasan ang hamong ito, binuo ng Qedma ang Quantum Error Suppression and Error Mitigation (QESEM), na seamlessly na naka-integrate sa IBM Quantum Platform bilang isang [Qiskit Function](/guides/functions).

Gamit ang QESEM, maaaring patakbuhin ng mga gumagamit ang kanilang mga quantum circuit sa maingay na QPU para makakuha ng lubos na tumpak na mga resulta na walang error, na may napaka-episyenteng gastos sa QPU time, malapit sa mga pundamental na hangganan. Para magawa ito, ginagamit ng QESEM ang isang hanay ng mga proprietary na pamamaraan na binuo ng Qedma para sa pag-characterize at pagbabawas ng mga error. Kasama sa mga pamamaraan ng pagbabawas ng error ang gate optimization, noise-aware transpilation, error suppression (ES), at unbiased error mitigation (EM). Sa kombinasyong ito ng mga characterization-based na pamamaraan, nakakamit ng mga gumagamit ang maaasahan at error-free na mga resulta para sa mga generic na large-volume quantum circuit, na nagbubukas ng mga application na hindi maisasagawa nang iba.

Para sa buong paglalarawan ng mga pinagbabatayan na bahagi, pati na rin ang isang utility-scale na demonstrasyon, tingnan ang papel na [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Paglalarawan
Maaari mong gamitin ang QESEM function ng Qedma para madaling i-estimate at i-execute ang iyong mga circuit na may error suppression at mitigation, na nakakamit ng mas malalaking circuit volume at mas mataas na katumpakan. Para gamitin ang QESEM, magbibigay ka ng isang quantum circuit, isang hanay ng mga observable na susukat, isang target na statistical accuracy para sa bawat observable, at isang piniling QPU. Bago patakbuhin ang circuit sa target na katumpakan, maaari mong i-estimate ang kinakailangang QPU time batay sa isang analytical na kalkulasyon na hindi nangangailangan ng circuit execution. Kapag nasiyahan ka na sa QPU time estimation, maaari mo nang i-execute ang circuit gamit ang QESEM.

Kapag nag-execute ka ng circuit, nagpapatakbo ang QESEM ng device characterization protocol na akma sa iyong circuit, na nagbubunga ng maaasahang noise model para sa mga error na nangyayari sa circuit. Batay sa characterization, unang ipinapatupad ng QESEM ang noise-aware transpilation para i-map ang input circuit sa isang hanay ng mga physical qubit at gate, na nagpapababa ng ingay na nakakaapekto sa target na observable. Kasama dito ang mga natively available na gate (CX/CZ sa IBM&reg; na mga device), pati na rin ang mga karagdagang gate na na-optimize ng QESEM, na bumubuo ng extended gate set ng QESEM. Nagpapatakbo rin ang QESEM ng isang hanay ng mga characterization-based na ES at EM circuit sa QPU at kinokolekta ang kanilang mga resulta ng pagsukat. Pagkatapos ay klasikal na pinoproseso ang mga ito para magbigay ng unbiased na expected value at error bar para sa bawat observable, na naaayon sa hiniling na katumpakan.

![Pangkalahatang-ideya ng Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Napatunayang nagbibigay ang QESEM ng mga resulta na may mataas na katumpakan para sa iba't ibang quantum application at sa pinakamalaking circuit volume na makakamit ngayon. Nag-aalok ang QESEM ng mga sumusunod na feature para sa mga gumagamit, na ipinapakita sa seksyon ng mga benchmark sa ibaba:
-	**Garantisadong katumpakan:** Nagpi-print ang QESEM ng mga unbiased na pagtatantya para sa mga expected value ng mga observable. Ang EM method nito ay may mga theoretical na garantiya na — kasama ang cutting-edge na characterization ng Qedma — tinitiyak na ang mitigation ay nagko-converge sa noiseless na output ng circuit hanggang sa user-specified na katumpakan. Kumpara sa maraming heuristic na EM method na madaling magkaroon ng sistematikong error o bias, ang garantisadong katumpakan ng QESEM ay mahalaga para matiyak ang maaasahang mga resulta sa mga generic na quantum circuit at observable.
-	**Scalability sa malalaking QPU:** Ang QPU time ng QESEM ay depende sa circuit volume, ngunit independyente sa bilang ng mga qubit. Naipakita na ng Qedma ang QESEM sa pinakamalaking quantum device available ngayon, kabilang ang IBM Quantum 127-qubit Eagle at 133-qubit Heron na mga device.
-	**Application-agnostic:** Napatunayan na ang QESEM sa iba't ibang application, kabilang ang Hamiltonian simulation, VQE, QAOA, at amplitude estimation. Maaaring mag-input ang mga gumagamit ng anumang quantum circuit at observable na susukat, at makakuha ng tumpak na mga resulta na walang error. Ang mga limitasyon lamang ay idinidikta ng mga hardware specification at inilaan na QPU time, na nagtatakda ng naa-access na circuit volume at output accuracy. Kumpara rito, maraming solusyon sa pagbabawas ng error ang application-specific o gumagamit ng hindi kontroladong heuristic, na ginagawa silang hindi angkop para sa mga generic na quantum circuit at application.
-  **Extended gate set:** Sinusuportahan ng QESEM ang mga fractional-angle gate, at nagbibigay ng Qedma-optimized fractional-angle $Rzz(\theta)$ gate sa mga IBM Quantum Eagle device. Ang extended gate set na ito ay nagpapahintulot ng mas episyenteng compilation at nagbubukas ng mga circuit volume na mas malaki ng hanggang 2x kumpara sa default na CX/CZ compilation.
-	**Multibase observables:** Sinusuportahan ng QESEM ang mga input observable na binubuo ng maraming non-commuting na Pauli string, tulad ng mga generic na Hamiltonian. Ang pagpili ng mga measurement basis at ang optimization ng QPU resource allocation (shots at circuits) ay awtomatikong ginagawa ng QESEM para maibaba ang kinakailangang QPU time para sa hiniling na katumpakan. Ang optimization na ito, na isinasaalang-alang ang hardware fidelity at execution rate, ay nagpapahintulot sa iyo na magpatakbo ng mas malalim na circuit at makamit ang mas mataas na katumpakan.
## Mga Benchmark
Nasubukan na ang QESEM sa malawak na iba't ibang kaso ng paggamit at application. Makakatulong ang mga sumusunod na halimbawa sa iyo sa pag-assess kung anong uri ng workload ang maaari mong patakbuhin gamit ang QESEM.

Isang mahalagang sukatan para ma-quantify ang kahirapan ng parehong error mitigation at classical simulation para sa isang partikular na circuit at observable ay ang **active volume**: ang bilang ng mga CNOT gate na nakakaapekto sa observable sa circuit. Ang active volume ay depende sa depth at width ng circuit, sa timbang ng observable, at sa estruktura ng circuit, na nagtatakda ng light cone ng observable. Para sa karagdagang detalye, tingnan ang talumpati mula sa [2024 IBM Quantum Summit](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). Nagbibigay ang QESEM ng partikular na malaking halaga sa high-volume regime, na nagbibigay ng maaasahang mga resulta para sa mga generic na circuit at observable.

![Active volume](../docs/images/guides/qedma-qesem/active_volume.svg)

| Application    | Bilang ng qubit | Device | Paglalarawan ng circuit | Katumpakan | Kabuuang oras | Paggamit ng runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 total layers, 9 measurement bases, 1D chain                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 3 steps, 2D heavy-hex topology                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 unique layers x 8 steps, 2D heavy-hex topology                     | 97%      | 116 min      | 23 min          |
| Trotterized Hamiltonian simulation   | 40  | Eagle (r3)            | 2 unique layers x 10 Trotter steps, 1D chain                    | 97%      | 3 hours     | 25 min         |
| Trotterized Hamiltonian simulation   | 119  | Eagle (r3)           | 3 unique layers x 9 Trotter steps, 2D heavy-hex topology                    | 95%      | 6.5 hours     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 unique layers x 15 steps, 2D heavy-hex topology                    | 99%      | 52 min             | 9 min           |

Ang katumpakan dito ay sinusukat kumpara sa ideal na halaga ng observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, kung saan ang '$\epsilon$' ay ang absolute na katumpakan ng mitigation (itinakda ng input ng gumagamit), at ang $\langle O \rangle_{ideal}$ ay ang observable sa noiseless na circuit.
Ang 'Runtime usage' ay sumusukat ng paggamit ng benchmark sa batch mode (kabuuan ng paggamit ng mga indibidwal na job), habang ang 'total time' ay sumusukat ng paggamit sa session mode (experiment wall time), na kinabibilangan ng karagdagang classical at communication time. Available ang QESEM para sa execution sa parehong mode, para magamit ng mga gumagamit ang kanilang mga available na mapagkukunan nang pinakamabuti.

Ang mga 28-qubit Kicked Ising circuit ay nagsi-simulate ng Discrete Time Quasicrystal na pinag-aralan ni Shinjo et al. (tingnan ang [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) at [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) sa tatlong magkakaugnay na loop ng ibm_kawasaki. Ang mga parameter ng circuit na ginamit dito ay $(\theta_x, \theta_z) = (0.9 \pi, 0)$, na may ferromagnetic na paunang estado $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Ang sinusukat na observable ay ang absolute na halaga ng magnetization $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Ang utility-scale na Kicked Ising experiment ay pinatakbo sa 136 pinakamahusay na qubit ng ibm_fez; ang partikular na benchmark na ito ay pinatakbo sa Clifford angle na $(\theta_x, \theta_z) = (\pi, 0)$, kung saan ang active volume ay mabagal na lumalaki kasabay ng circuit depth, na — kasama ang mataas na fidelity ng device — nagbibigay-daan sa mataas na katumpakan sa maikling runtime.

Ang mga Trotterized Hamiltonian simulation circuit ay para sa isang Transverse-Field Ising model sa mga fractional angle: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ at $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ ayon sa pagkakasunod (tingnan ang [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Ang utility-scale na circuit ay pinatakbo sa 119 pinakamahusay na qubit ng ibm_brisbane, habang ang 40-qubit na eksperimento ay pinatakbo sa pinakamahusay na available na chain. Ang katumpakan ay iniulat para sa magnetization; nakamit din ang mga resulta na may mataas na katumpakan para sa mga observable na may mas mataas na timbang.

Ang VQE circuit ay binuo kasama ang mga mananaliksik mula sa Center for Quantum Technology and Applications sa Deutsches Elektronen-Synchrotron (DESY). Ang target na observable dito ay isang Hamiltonian na binubuo ng malaking bilang ng mga non-commuting na Pauli string, na binibigyang-diin ang optimized na performance ng QESEM para sa multi-basis observable. Ang mitigation ay inilapat sa isang classically-optimized na ansatz; kahit hindi pa nailalathala ang mga resultang ito, makukuha ang parehong kalidad ng mga resulta para sa iba't ibang circuit na may magkaparehong istruktura.
## Pagsisimula
Mag-authenticate gamit ang iyong [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/), at piliin ang QESEM Qiskit Function tulad ng sumusunod. (Ipinapalagay ng snippet na ito na [na-save mo na ang iyong account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na environment.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Halimbawa
Para magsimula, subukan ang pangunahing halimbawang ito ng pag-estimate ng kinakailangang QPU time para patakbuhin ang QESEM para sa isang partikular na `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Ang sumusunod na halimbawa ay nagpapatakbo ng isang QESEM job:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Maaari mong gamitin ang mga pamilyar na Qiskit Serverless API para suriin ang status ng iyong Qiskit Function workload o makuha ang mga resulta:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Mga parameter ng function
| Pangalan |  Uri | Paglalarawan | Kinakailangan | Default |  Halimbawa |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Ito ang pangunahing input. Ang `Pub` ay naglalaman ng 2-4 na elemento: isang circuit, isa o higit pang observable, 0 o isang hanay ng mga halaga ng parameter, at isang opsyonal na katumpakan. Kung hindi tinukoy ang katumpakan, gagamitin ang `default_precision` mula sa `options` |  Oo|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string| Pangalan ng backend na gagamitin |Hindi | Kukuha ang QESEM ng pinaka-hindi abala na device na iniulat ng IBM| `"ibm_fez"`|
| `instance` | string|  Ang cloud resource name ng instance na gagamitin sa format na iyon |  Hindi |  N/A | `"CRN"`  |
| `options` | dictionary | Mga input na opsyon. Tingnan ang seksyon ng **Options** para sa karagdagang detalye. | Hindi |  Tingnan ang seksyon ng **Options** para sa mga detalye.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Mga Opsyon
| Opsyon |  Mga Pagpipilian | Paglalarawan | Default |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Kapag itinakda, ang QESEM job ay magkalkula lamang ng QPU time estimation. Pakitingnan ang sumusunod na paglalarawan para sa karagdagang detalye. Kapag itinakda sa None, ang circuit ay ie-execute gamit ang QESEM| None |
|`default_precision` | 0 < float | Ilalapat sa mga `pub` na walang katumpakan. Ang katumpakan ay nagpapahiwatig ng katanggap-tanggap na error sa mga expected value ng mga observable sa absolute na halaga. Ibig sabihin, ang QPU runtime para sa mitigation ay matutukoy para magbigay ng mga output na halaga para sa lahat ng observable na interesado na nahuhulog sa loob ng `1σ` na confidence interval ng target na katumpakan. Kung maraming observable ang ibinigay, magpapatuloy ang mitigation hanggang maabot ang target na katumpakan para sa bawat isa sa mga input na observable. | 0.02|
|`max_execution_time` | 0 < integer < 28,800 (8 oras)| Nagpapahintulot sa iyo na limitahan ang QPU time, na tinukoy sa segundo, na gagamitin para sa buong proseso ng QESEM. Pakitingnan ang karagdagang detalye sa ibaba.| 3,600 (isang oras)|
| `transpilation_level` | 0 / 1 | Tingnan ang paglalarawan sa ibaba | 1|
| `execution_mode` | `"session"` / `"batch"` | Tingnan ang sumusunod na paglalarawan | "batch"|

> **Caution:** Nagbabago ang QPU time estimation mula sa isang backend patungo sa isa pa. Kaya naman, kapag ine-execute ang QESEM function, tiyaking patakbuhin ito sa parehong backend na pinili nang makuha ang QPU time estimation.

> **Note:** Matatapos ang pagpapatakbo ng QESEM kapag naabot na nito ang target na katumpakan o kapag naabot na ang `max_execution_time`, alinman ang mauna.

- `estimate_time_only` — Ang flag na ito ay nagpapahintulot sa mga gumagamit na makakuha ng pagtatantya ng QPU time na kailangan para i-execute ang circuit gamit ang QESEM.
    - Kung itinakda sa `"analytical"`, isang upper bound ng QPU time ang kinakalkula nang hindi gumagamit ng anumang QPU. Ang estimasyong ito ay may 30-minutong resolusyon (halimbawa, 30 minuto, 60 minuto, 90 minuto, at iba pa). Karaniwang pessimistic ito, at makukuha lamang para sa mga single Pauli observable o kabuuan ng mga Pauli na walang intersecting na suporta (halimbawa, Z0+Z1). Pangunahing kapaki-pakinabang ito para sa paghahambing ng mga antas ng kumplikasyon ng iba't ibang parameter na ibinibigay ng gumagamit (circuit, katumpakan, at iba pa).
    - Para makakuha ng mas tumpak na QPU time estimation, itakda ang flag na ito sa `"empirical"`. Kahit na nangangailangan ng pagpapatakbo ng maliit na bilang ng mga circuit ang opsyong ito, nagbibigay ito ng mas tumpak na QPU time estimation. Ang estimasyong ito ay may 5-minutong resolusyon (halimbawa, 20 minuto, 25 minuto, 30 minuto, at iba pa). Maaaring piliin ng gumagamit na patakbuhin ang empirical time estimation sa alinmang batch o session mode. Para sa karagdagang detalye, pakitingnan ang paglalarawan ng `execution_mode`. Halimbawa, sa batch mode, ang empirical time estimation ay gagamitin ng wala pang 10 minuto ng QPU time.

- `max_execution_time`: Nagpapahintulot sa iyo na limitahan ang QPU time, na tinukoy sa segundo, na gagamitin para sa buong proseso ng QESEM. Dahil ang panghuling QPU time na kailangan para maabot ang target na katumpakan ay dinamikong natutukoy sa panahon ng QESEM job, ang parameter na ito ay nagpapahintulot sa iyo na limitahan ang gastos ng eksperimento. Kung ang dinamikong tinutukoy na QPU time ay mas maikli kaysa sa oras na inilaan ng gumagamit, hindi maaapektuhan ng parameter na ito ang eksperimento. Ang parameter na `max_execution_time` ay partikular na kapaki-pakinabang sa mga kaso kung saan ang analytical na time estimate na ibinibigay ng QESEM bago magsimula ang job ay masyadong pessimistic at nais ng gumagamit na simulan ang isang mitigation job kahit na ganoon. Pagkatapos maabot ang limitasyon ng oras, titigil ang QESEM sa pagpapadala ng mga bagong circuit. Ang mga circuit na naipadala na ay magpapatuloy sa pagpapatakbo (kaya ang kabuuang oras ay maaaring lumampas sa limitasyon ng hanggang 30 minuto), at matatanggap ng gumagamit ang mga naprosesong resulta mula sa mga circuit na pinatakbo hanggang sa puntong iyon. Kung nais mong mag-apply ng QPU time limit na mas maikli kaysa sa analytical time estimate, kumonsulta sa Qedma para makakuha ng pagtatantya ng katumpakan na makakamit sa loob ng limitasyon ng oras.

- `transpilation_level`: Pagkatapos isumite ang isang circuit sa QESEM, awtomatiko itong naghahanda ng ilang alternatibong circuit transpilation at pipili ng isa na nagpapababa ng QPU time. Halimbawa, maaaring gumamit ang mga alternatibong transpilation ng Qedma-optimized fractional RZZ gate para mabawasan ang depth ng circuit. Syempre, lahat ng transpilation ay katumbas ng input circuit sa mga tuntunin ng kanilang ideal na output. Para mag-exert ng mas maraming kontrol sa circuit transpilation, itakda ang transpilation level sa `options`. Habang ang `"transpilation_level": 1` ay naaayon sa default na gawi na inilalarawan sa itaas, ang `"transpilation_level": 0` ay kinabibilangan lamang ng minimal na mga pagbabago na kailangan sa orihinal na circuit; halimbawa, 'layerification' — ang organisasyon ng mga circuit operation sa mga 'layer' ng magkasamang two-qubit gate. Tandaan na ang awtomatikong hardware-mapping sa mga high-fidelity na qubit ay ilalapat sa anumang kaso.

| transpilation_level | Paglalarawan |
|:-:|:--|
| `1` | Default na QESEM transpilation. Naghahanda ng ilang alternatibong transpilation at pumipili ng isa na nagpapababa ng QPU time. Maaaring mabago ang mga barrier sa hakbang ng layerification. |
| `0` | Minimal na transpilation: ang mitigated na circuit ay magiging katulad ng istruktura ng input circuit. Ang mga circuit na ibinigay sa level 0 ay dapat tumugma sa koneksyon ng device at dapat na tinukoy sa mga sumusunod na gate: CX, Rzz(α), at mga standard na single-qubit gate (U, x, sx, rz, at iba pa). Igagalang ang mga barrier sa hakbang ng layerification. |

- `execution_mode` — Maaaring piliin ng gumagamit na patakbuhin ang QESEM job sa alinmang dedicated na [IBM session](/guides/execution-modes#session-mode) o sa maraming [IBM batch](/guides/execution-modes#batch-mode):
    -   **Session Mode**: Mas mahal ang mga session ngunit nagreresulta sa mas maikli na time-to-result. Kapag nagsimula na ang session, ang QPU ay nakalaan nang eksklusibo para sa QESEM job. Ang pagkalkula ng paggamit ay kinabibilangan ng parehong oras na ginugol sa QPU execution at ang mga kasamang classical computation (na ginagawa ng QESEM at IBM). Ang QESEM Qiskit Function ay nag-aayos ng paglikha at pagsasara ng session nang awtomatiko. Para sa mga gumagamit na may walang limitasyong access sa mga QPU (halimbawa, on-premises na setup), inirerekumenda ang session mode para sa mas mabilis na QESEM execution.
    -   **Batch Mode**: Sa batch mode, ang QPU ay inilalabas sa panahon ng mga classical computation, na nagreresulta sa mas mababang paggamit ng QPU. Dahil ang mga batch job ay karaniwang sumasaklaw ng mas mahabang panahon, mas malaki ang panganib ng hardware drift; isinasama ng QESEM ang mga hakbang para matukoy at mabayaran ang mga drift, pinapanatiling maaasahan ang mga resulta sa mahabang pagpapatakbo.

> **Note:** Ang mga barrier operation ay karaniwang ginagamit para tukuyin ang mga layer ng two-qubit gate sa mga quantum circuit. Sa level 0, pinapanatili ng QESEM ang mga layer na tinukoy ng mga barrier. Sa level 1, ang mga layer na tinukoy ng mga barrier ay itinuturing bilang isang alternatibong transpilation kapag pinipigilan ang QPU time.
### Mga Output
Ang output ng isang Circuit function ay isang [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), na naglalaman ng dalawang field:

- Isang [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) na object. Maaari itong direktang ma-index mula sa `PrimitiveResult`.

- Job-level metadata.

Ang bawat `PubResult` ay naglalaman ng `data` at `metadata` na field.

- Ang `data` field ay naglalaman ng kahit isang array ng mga expected value (`PubResult.data.evs`) at isang array ng mga standard error (`PubResult.data.stds`). Maaari rin itong maglaman ng karagdagang data, depende sa mga opsyong ginamit.

- Ang `metadata` field ay naglalaman ng PUB-level na metadata (`PubResult.metadata`).

Ang sumusunod na code snippet ay nagpapaliwanag kung paano kunin ang QPU time estimation (kapag itinakda ang `estimate_time_only`):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Ang sumusunod na code snippet ay nagpapakita kung paano kunin ang mga resulta ng mitigation (kapag hindi itinakda ang `estimate_time_only`) at mga execution metric. Naglalaman ang mga ito ng mahahalagang data na nagpapahintulot sa mas malalim na pag-unawa kung paano nakakaapekto ang iba't ibang parameter sa QESEM execution. Maaari rin itong maging kaugnay kapag nagsusulat ng papel batay sa iyong pananaliksik.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Kunin ang mga mensahe ng error
Kung ang status ng iyong workload ay ERROR, gamitin ang job.result() para kunin ang mensahe ng error tulad ng sumusunod:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>